# Sse

> SSE parsing helpers.

In [ ]:
#| default_exp sse

## Imports

In [ ]:
#| export
from dataclasses import dataclass
import httpx,json
from fastcore.utils import *

from fastspec.errors import *

In [ ]:
#| hide
from cachy.core import enable_cachy
from fastcore.test import *

In [ ]:
enable_cachy()

### `SSEvent`

A parsed SSE event

In [ ]:
#| export
@dataclass(frozen=True)
class SSEvent:
    "A parsed SSE event."
    event: str
    data: str
    id: str = None
    retry: int = None

### `parse_sse_lines`

Parses synchronous SSE line streams into `SSEvent` objects.

**Why it exists**
- Provides a minimal framing parser reusable in tests and non-async contexts.

**Connections**
- Conceptual counterpart to `aiter_sse` used in async transport paths.

Ok let's send api calls for each provider and parse few lines

Write examples for each similar to how we did in errors dialog

This is not exported as it's not needed since we keep the higher level api async only for simplicity:

In [ ]:
def parse_sse_lines(lines):
    "Parse SSE line stream into events."
    event, data, event_id, retry = None, [], None, None
    for raw in lines:
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)

In [ ]:
with httpx.stream("POST", "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json={"model": "gpt-4o-mini", "stream": True, "max_tokens": 20,
          "messages": [{"role": "user", "content": "Say hello"}]}
) as r:
    oai_lines = list(r.iter_lines())
oai_lines[:10]

['data: {"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"role":"assistant","content":"","refusal":null},"logprobs":null,"finish_reason":null}],"obfuscation":"k0Usqv"}',
 '',
 'data: {"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"Hello"},"logprobs":null,"finish_reason":null}],"obfuscation":"vBg"}',
 '',
 'data: {"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"!"},"logprobs":null,"finish_reason":null}],"obfuscation":"DlpNx9E"}',
 '',
 'dat

In [ ]:
with httpx.stream("POST", "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "stream": True, "max_tokens": 20,
          "messages": [{"role": "user", "content": "Say hello"}]}
) as r:
    ant_lines = list(r.iter_lines())
ant_lines[:10]

['event: message_start',
 'data: {"type":"message_start","message":{"model":"claude-sonnet-4-20250514","id":"msg_015ZCvz5Gqn8CHzRub5DfZJ8","type":"message","role":"assistant","content":[],"stop_reason":null,"stop_sequence":null,"usage":{"input_tokens":9,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":4,"service_tier":"standard","inference_geo":"not_available"}} }',
 '',
 'event: content_block_start',
 'data: {"type":"content_block_start","index":0,"content_block":{"type":"text","text":""}               }',
 '',
 'event: ping',
 'data: {"type": "ping"}',
 '',
 'event: content_block_delta']

In [ ]:
with httpx.stream("POST",
    f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:streamGenerateContent?alt=sse&key={os.environ['GEMINI_API_KEY']}",
    json={"contents": [{"parts": [{"text": "Say hello"}]}]}
) as r:
    gem_lines = list(r.iter_lines())
gem_lines[:10]

['data: {"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}',
 '']

In [ ]:
@patch
def data_json(self: SSEvent): 
    try: return json.loads(self.data)
    except: return self.data

@patch
def data_obj(self: SSEvent): return dict2obj(self.data_json())

In [ ]:
oai_sse = L(parse_sse_lines(oai_lines))
ant_sse = L(parse_sse_lines(ant_lines))
gem_sse = L(parse_sse_lines(gem_lines))

In [ ]:
oai_sse[:3]

[SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"role":"assistant","content":"","refusal":null},"logprobs":null,"finish_reason":null}],"obfuscation":"k0Usqv"}', id=None, retry=None), SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"Hello"},"logprobs":null,"finish_reason":null}],"obfuscation":"vBg"}', id=None, retry=None), SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"c

In [ ]:
oai_sse[-1]

SSEvent(event=None, data='[DONE]', id=None, retry=None)

In [ ]:
oai_sse[0].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': '', 'refusal': {}}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'k0Usqv',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse[1].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'content': 'Hello'}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'vBg',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse[-1].data_obj()

'[DONE]'

In [ ]:
ant_sse[:3]

[SSEvent(event='message_start', data='{"type":"message_start","message":{"model":"claude-sonnet-4-20250514","id":"msg_015ZCvz5Gqn8CHzRub5DfZJ8","type":"message","role":"assistant","content":[],"stop_reason":null,"stop_sequence":null,"usage":{"input_tokens":9,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":4,"service_tier":"standard","inference_geo":"not_available"}} }', id=None, retry=None), SSEvent(event='content_block_start', data='{"type":"content_block_start","index":0,"content_block":{"type":"text","text":""}               }', id=None, retry=None), SSEvent(event='ping', data='{"type": "ping"}', id=None, retry=None)]

In [ ]:
ant_sse[-1]

SSEvent(event='message_stop', data='{"type":"message_stop"          }', id=None, retry=None)

In [ ]:
ant_sse[0].data_obj()

```python
{ 'message': { 'content': [],
               'id': 'msg_015ZCvz5Gqn8CHzRub5DfZJ8',
               'model': 'claude-sonnet-4-20250514',
               'role': 'assistant',
               'stop_reason': {},
               'stop_sequence': {},
               'type': 'message',
               'usage': { 'cache_creation': { 'ephemeral_1h_input_tokens': 0,
                                              'ephemeral_5m_input_tokens': 0},
                          'cache_creation_input_tokens': 0,
                          'cache_read_input_tokens': 0,
                          'inference_geo': 'not_available',
                          'input_tokens': 9,
                          'output_tokens': 4,
                          'service_tier': 'standard'}},
  'type': 'message_start'}
```

In [ ]:
ant_sse[1].data_obj()

```python
{ 'content_block': {'text': '', 'type': 'text'},
  'index': 0,
  'type': 'content_block_start'}
```

In [ ]:
ant_sse[2].data_obj()

```python
{'type': 'ping'}
```

In [ ]:
ant_sse[3].data_obj()

```python
{ 'delta': {'text': 'Hello! How are', 'type': 'text_delta'},
  'index': 0,
  'type': 'content_block_delta'}
```

In [ ]:
ant_sse[-1].data_obj()

```python
{'type': 'message_stop'}
```

In [ ]:
gem_sse[:3]

[SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)]

In [ ]:
gem_sse[-1]

SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)

In [ ]:
gem_sse[0].data_obj()

```python
{ 'candidates': [{'content': {'parts': [{'text': 'Hello!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}],
  'modelVersion': 'gemini-2.5-flash',
  'responseId': 'E6bLabD3FqvujrEP1fel0A4',
  'usageMetadata': { 'candidatesTokenCount': 2,
                     'promptTokenCount': 2,
                     'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2}],
                     'thoughtsTokenCount': 15,
                     'totalTokenCount': 19}}
```

### `aiter_sse`

Async SSE parser over an `httpx` streamed response

In [ ]:
#| export
async def aiter_sse(response: httpx.Response):
    "Async SSE parser from an httpx streamed response."
    event, data, event_id, retry = None, [], None, None
    async for raw in response.aiter_lines():
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)

In [ ]:
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST", "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
        json={"model": "gpt-4o-mini", "stream": True, "max_tokens": 20,
              "messages": [{"role": "user", "content": "Say hello"}]}
    ) as r:
        oai_sse2 = [e async for e in aiter_sse(r)]
oai_sse2[:3]

[SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"role":"assistant","content":"","refusal":null},"logprobs":null,"finish_reason":null}],"obfuscation":"k0Usqv"}', id=None, retry=None),
 SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{"content":"Hello"},"logprobs":null,"finish_reason":null}],"obfuscation":"vBg"}', id=None, retry=None),
 SSEvent(event=None, data='{"id":"chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0","object":"chat.completion.chunk","created":1774953619,"model":"gpt-4o-mini-2024-07-18","service_tier":"default","system_fingerprint":"fp_dfc054be05","choices":[{"index":0,"delta":{

In [ ]:
oai_sse2[-1]

SSEvent(event=None, data='[DONE]', id=None, retry=None)

In [ ]:
oai_sse2[0].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': '', 'refusal': {}}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'k0Usqv',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse2[1].data_obj()

```python
{ 'choices': [{'index': 0, 'delta': {'content': 'Hello'}, 'logprobs': {}, 'finish_reason': {}}],
  'created': 1774953619,
  'id': 'chatcmpl-DPQo7UACCo4OMnw0zNOb9FvHo8bw0',
  'model': 'gpt-4o-mini-2024-07-18',
  'obfuscation': 'vBg',
  'object': 'chat.completion.chunk',
  'service_tier': 'default',
  'system_fingerprint': 'fp_dfc054be05'}
```

In [ ]:
oai_sse2[-1].data_obj()

'[DONE]'

In [ ]:
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST", "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"},
        json={"model": "claude-sonnet-4-20250514", "stream": True, "max_tokens": 20,
              "messages": [{"role": "user", "content": "Say hello"}]}
    ) as r:
        ant_sse2 = [e async for e in aiter_sse(r)]
ant_sse2[:3]

[SSEvent(event='message_start', data='{"type":"message_start","message":{"model":"claude-sonnet-4-20250514","id":"msg_015ZCvz5Gqn8CHzRub5DfZJ8","type":"message","role":"assistant","content":[],"stop_reason":null,"stop_sequence":null,"usage":{"input_tokens":9,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":4,"service_tier":"standard","inference_geo":"not_available"}} }', id=None, retry=None),
 SSEvent(event='content_block_start', data='{"type":"content_block_start","index":0,"content_block":{"type":"text","text":""}               }', id=None, retry=None),
 SSEvent(event='ping', data='{"type": "ping"}', id=None, retry=None)]

In [ ]:
ant_sse2[-1]

SSEvent(event='message_stop', data='{"type":"message_stop"          }', id=None, retry=None)

In [ ]:
ant_sse2[0].data_obj()

```python
{ 'message': { 'content': [],
               'id': 'msg_015ZCvz5Gqn8CHzRub5DfZJ8',
               'model': 'claude-sonnet-4-20250514',
               'role': 'assistant',
               'stop_reason': {},
               'stop_sequence': {},
               'type': 'message',
               'usage': { 'cache_creation': { 'ephemeral_1h_input_tokens': 0,
                                              'ephemeral_5m_input_tokens': 0},
                          'cache_creation_input_tokens': 0,
                          'cache_read_input_tokens': 0,
                          'inference_geo': 'not_available',
                          'input_tokens': 9,
                          'output_tokens': 4,
                          'service_tier': 'standard'}},
  'type': 'message_start'}
```

In [ ]:
ant_sse2[1].data_obj()

```python
{ 'content_block': {'text': '', 'type': 'text'},
  'index': 0,
  'type': 'content_block_start'}
```

In [ ]:
ant_sse2[2].data_obj()

```python
{'type': 'ping'}
```

In [ ]:
ant_sse2[3].data_obj()

```python
{ 'delta': {'text': 'Hello! How are', 'type': 'text_delta'},
  'index': 0,
  'type': 'content_block_delta'}
```

In [ ]:
ant_sse2[-1].data_obj()

```python
{'type': 'message_stop'}
```

In [ ]:
async with httpx.AsyncClient() as cli:
    async with cli.stream("POST",
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:streamGenerateContent?alt=sse&key={os.environ['GEMINI_API_KEY']}",
        json={"contents": [{"parts": [{"text": "Say hello"}]}]}
    ) as r:
        gem_sse2 = [e async for e in aiter_sse(r)]
gem_sse2[:3]

[SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)]

In [ ]:
gem_sse2[-1]

SSEvent(event=None, data='{"candidates": [{"content": {"parts": [{"text": "Hello!"}],"role": "model"},"finishReason": "STOP","index": 0}],"usageMetadata": {"promptTokenCount": 2,"candidatesTokenCount": 2,"totalTokenCount": 19,"promptTokensDetails": [{"modality": "TEXT","tokenCount": 2}],"thoughtsTokenCount": 15},"modelVersion": "gemini-2.5-flash","responseId": "E6bLabD3FqvujrEP1fel0A4"}', id=None, retry=None)

In [ ]:
gem_sse2[0].data_obj()

```python
{ 'candidates': [{'content': {'parts': [{'text': 'Hello!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}],
  'modelVersion': 'gemini-2.5-flash',
  'responseId': 'E6bLabD3FqvujrEP1fel0A4',
  'usageMetadata': { 'candidatesTokenCount': 2,
                     'promptTokenCount': 2,
                     'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2}],
                     'thoughtsTokenCount': 15,
                     'totalTokenCount': 19}}
```

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()